In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

  Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.7.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.8 MB)
Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.7 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached stevedore-5.7.0-py3-none-any.whl (54 kB)
Using cached sympy-1.14.0-py3-none-any.whl

In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
## defining constants used throughout the protocol

root3 = math.sqrt(3)
root2 = math.sqrt(2)

denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2) 


# This is the matrix to map states in the W basis to the standard basis,
# where the W basis is defined by:
# W = 1/root2(X + Z)
W_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                        [  1 / denom2 , (root2 - 1) / denom2 ] ]
W_transform = Operator(W_transform_matrix) 

# This is the matrix to map the standard basis to the W basis, which is used in Eve's
# intercept-send attack
W_inverse_matrix = [[(1 - root2)/denom2, (1 + root2)/denom1],
                    [1/denom2, 1/denom1]]
W_inverse = Operator(W_inverse_matrix)

# This is the matrix to map states in the V basis to the standard basis,
# where the V basis is defined by:
# V = 1/root2(X - Z)
V_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                        [ -1 / denom2 , (root2 - 1) / denom2 ] ]
V_transform = Operator(V_transform_matrix) 

In [4]:
## Defines auxilliary functions that are used throughout the protocol

def entangledPair():
    q = QuantumCircuit(2, 2)
    q.h(1)
    q.cx(1,0)
    q.x(0)
    q.z(1)
    return q

# this creates a circuit that can be used to give a uniform random integer
# between 0 and 2 (inclusive). The circuit first applies a unitary
# operator that converts |0> -> (1/root3)|0> + (root2/root3)|1>. It then
# applies the Hadamard operator to the second qubit, giving 1/root2(|0> + |1>)
def random_3():
    q = QuantumCircuit(2)
    random_matrix = [[ 1 / root3, - root2 / root3 ],
                     [ root2 / root3, 1 / root3 ]]
    q.unitary(random_matrix, [1])
    q.h(0)
    q.measure_all()

    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    return list(counts.keys())[0]
    

# This is a helper function that runs the circuit defined in random_3, to get a uniform
# random integer between 1 and 3. This random integer determines the random basis
# that Alice and Bob measure their qubit in

# Measuring the first qubit, there is a 1/3 chance that the first is 0, so if this occurs
# the result is 0. There is a 2/3 chance that the result is 1, and in this case
# we look to the second qubit. The result of measuring the second qubit is 0 or 1   
# with 1/2 chance each, meaning that in both qubits the measurements 10 and 11 each occur
# on average 1/3 of the time. In this case, the measurement result is converted to
# an integer (either 1 or 2), by summing the individual bits. 
# 1 is added to the result to keep the basis notation consistent with the assignment sheet
def measurement_choice():
    choice = random_3()
   
    if choice[0] == '0':
        choice = 0
    else:
        choice = int(choice[0]) + int(choice[1])
        
    return choice + 1

# This helper function rotates Alice's qubit from the basis it is being measured in,
# to the standard basis
def alice_basis_rotation(q, alice_basis):
    if alice_basis == 1:
        q.h(1)

    elif alice_basis == 2:
        q.unitary(W_transform,[1])


# This helper function rotates Bob's qubit from the basis it is being measured in,
# to the standard basis
def bob_basis_rotation(q, bob_basis):

    if bob_basis == 1:
        q.unitary(W_transform, [0])


    elif bob_basis == 3:
        q.unitary(V_transform, [0])


# A helper function for running a circuit once and returning the result
def simulate_circuit(q):
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled)
    return list(counts.keys())[0]


# This implements an intercept resend attack, which takes the entangled pair of Alice and Bob,
# and a basis guess from Eve. Eve intercepts Alice's qubit, and measures it in the basis she
# has guessed. This is done by first converting to the standard basis, and then
# applying a normal measurement. 
# Note, that Eve can only guess the bases X and Z, since only qubits measured in these
# bases are used for the key.

# Eve then resets and manipulates Alice's qubit to reflect the result of her measurement,
# and converts the resulting qubit back to her chosen basis. This means that if Alice picks
# the same basis as Eve did, she is guaranteed to get the same result from measuring her qubit
def eve_intercept_resend(q, eve_basis):
    
    if eve_basis == 2:
        q.unitary(W_transform,[1])

    q.measure(1,1)
    
    if eve_basis == 2:
        q.unitary(W_inverse, [1])

In [5]:
## The following functions are all used for getting uniform random numbers from a given range
## These are used to determine how often (on average) Eve will attack the entangled pair 
## transmitted to Alice and Bob, and are passed into the main protocol function as a
## parameter. For example, passing random_4 into the protocol means that Eve will attack,
## on average, 1 in 4 entangled pairs.



# This achieves a random number between 0 and 1 by applying the Hadamard operator to a qubit,
# and measuring

# This function is also used for Eve's choice between the bases Z and W 
def random_2():
    q = QuantumCircuit(1)
    q.h(0)
    q.measure_all()

    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled)
    result = list(counts.keys())[0]
    return int(result)

    
# Achieves a random number between 0 and 3 by applying the Hadamard to two qubits, and
# measuring
def random_4():
    q = QuantumCircuit(2)
    q.h(0)
    q.h(1)
    q.measure_all()

    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled)
    result = list(counts.keys())[0]
    result = int(result[1]) + 2 * int(result[0])
    return result


# Passing this function in means Eve attacks every entangled pair
def constant_attack():
    return 1

# Passing this function in means that Eve does not attack any entangled pair
def no_attack():
    return -1

In [6]:
## The main protocol

# N : average length of key produced
# eve_random : the random function that determines if Eve attacks an entangled pair

def ekert_91_intercept_and_send(N, eve_random):
    
    alice_choices = []
    bob_choices = []
    eve_choices = []
    
    alice_results = []
    bob_results = []
    eve_results = []
    
    for i in range ((9 * N) // 2):
        q = entangledPair()
    
        alice_choice = measurement_choice()
        alice_choices.append(alice_choice)
        
        bob_choice = measurement_choice()
        bob_choices.append(bob_choice)

        # if the random function gives 1
        # e.g. random_2 -> 1/2 on average
        if eve_random() == 1:

            # random_2 gives {0,1}, so convert to {2,3} corresponding to a choice of {W,Z}
            eve_choice = random_2() + 2
            eve_choices.append(eve_choice)

            # measure Alice's qubit in chosen basis, and send on manipulated qubit to Alice
            eve_intercept_resend(q, eve_choice)

        else:
            eve_choices.append(-1)

        # converting Alice's qubit to her chosen basis
        alice_basis_rotation(q, alice_choice)

        # converting Bob's qubit to his chosen basis
        bob_basis_rotation(q, bob_choice)

        q.measure_all()

        result = simulate_circuit(q)

        ## Storing the results
        
        alice_results.append(int(result[0]))

        # flipping Bob's bit
        bob_results.append(1 - int(result[1]))

        # add Eve's "guess" to her list of results
        eve_results.append(int(result[3]))
    

    alice_key = []
    bob_key = []
    eve_key = []

    # counts how many bits of Alice's key that Eve knows
    eve_count = 0
    
    X_W = 0
    X_V = 0
    Z_W = 0
    Z_V = 0
    
    X_W_count = 0
    X_V_count = 0
    Z_W_count = 0
    Z_V_count = 0
 
    
    for i in range((9 * N) // 2):
    
        # Alice and Bob's choice of bases, and Eve's guess
        alice_choice = alice_choices[i]
        bob_choice = bob_choices[i]
        eve_choice = eve_choices[i]

        # Their results
        alice_bit = alice_results[i]
        bob_bit = bob_results[i]
        eve_bit =  eve_results[i]


        # if Alice and Bob are both measuring in the W basis
        if alice_choices[i] == 2 and bob_choices[i] == 1:
        
            alice_key.append(alice_bit)
            bob_key.append(bob_bit)

            # if Eve guessed right
            if eve_choice == 2:
                eve_key.append(eve_bit)
                eve_count += 1

            # otherwise fill with a blank
            else:
                eve_key.append(-1)

        # If Alice and Bob are both measuring in the Z basis
        elif alice_choices[i] == 3 and bob_choices[i] == 2:
            alice_key.append(alice_bit)
            bob_key.append(bob_bit)

            # if Eve guessed right
            if eve_choice == 3:
                eve_key.append(eve_bit)
                eve_count += 1

            # otherwise fill with a blank
            else:
                eve_key.append(-1)
                
        else:

            # converting from {0,1} -> {+1,-1}
            alice_bit = 1 - 2 * alice_bit
            bob_bit = 1 - 2 * bob_bit
            product = alice_bit * bob_bit

            # Alice chooses X and Bob chooses W
            if alice_choices[i] == 1 and bob_choices[i] == 1:
                X_W += product
                X_W_count += 1

            # Alice chooses X and Bob chooses V
            elif alice_choices[i] == 1 and bob_choices[i] == 3:
                X_V += product
                X_V_count += 1

            # Alice chooses Z and Bob chooses W 
            elif alice_choices[i] == 3 and bob_choices[i] == 1:
                Z_W += product
                Z_W_count += 1
                
            # Alice chooses Z and Bob chooses V
            elif alice_choices[i] == 3 and bob_choices[i] == 3:
                Z_V += product
                Z_V_count += 1
    
    X_W_average = X_W / X_W_count
    X_V_average = X_V / X_V_count
    Z_W_average = Z_W / Z_W_count
    Z_V_average = Z_V / Z_V_count
    
    s = abs(X_W_average - X_V_average + Z_W_average + Z_V_average)

    # Eve's attack sometimes will cause a mismatch in Bob and Alice's key
    mismatch_count = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
    
    return {
        "alice" : alice_key,
        "bob" : bob_key,
        "eve" : eve_key,
        "eve_count" : eve_count,
        "error_count" : mismatch_count,
        "s_value" : s
    }

In [10]:
# random_2 -> Eve attacks 1/2 of all pairs, 25% knowledge of key on average (due to guessing basis)
# measurement_choice -> Eve attacks 1/3 of all pairs, 16.6% knowledge of key on average
# random_4 -> Eve attacks 1/4 of all pairs, 12.5% knowledge of key on average
# constant_attack -> Eve attacks every pair, 50% knowledge of key on average
# no_attack -> Eve does not attack, same as no Ekert91_Plain

# The more Eve attacks, the more of Alice's key she knows. But also:
# * The more likely she is to be detected, and
# * The more mismatches in Alice and Bob's key - they likely would just redo the protocol

results = ekert_91_intercept_and_send(100, constant_attack)

# the number of bits that differ between Alice and Bob's key
error_count = results["error_count"]

key_length = len(results["alice"])
key_relative_error = error_count / key_length * 100

s_value = results["s_value"]
s_relative_error = abs(2 * root2 - s_value ) / (2 * root2) * 100

eve_count = results["eve_count"]

# proportion of Alice's key that Eve has knowledge of
eve_key_percentage = eve_count / key_length * 100

print(f"Key Length: {key_length}")
print(f"Relative Error in keys {key_relative_error:.2f}%")
print(f"Value of S: {s_value}")
print(f"Relative Error in S: {s_relative_error:.2f}%")
print(f"Number of bits of Alice key Eve knows: {eve_count}")
print(f"Percentage of key known to Eve: {eve_key_percentage:.2f}%")

# this is a naive function for calculating the probability of detecting Eve
# based on the calculated value of S. It is a simple exponential function, and
# the coefficient 9.5 is derived from trial and error, based on the fact that an S value
# of <= 2 should give certainty that there is an attacker present. 
# In a real life example, Alice and Bob could increase the value of this coefficient,
# which would predict an attacker for smaller deviations in S, useful when transmitting
# more sensitive information

difference = abs(2 * root2 - results["s_value"])
probability_function = math.exp(9.5 * difference)

# anything below ~ 2.35 gives certainty that an attacker is present
probability = probability_function if probability_function < 100 else 100
print(f"Probabilty of detecting Eve: {probability:.2f}%\n")

# This is to show that for every non-blank bit in Eve's key, it is the same as the corresponding
# bit in Alice's key (blanks are denoted as -1).

alice_key = results["alice"]
eve_key = results["eve"]
correct_eve = [eve_key[i] == alice_key[i] for i in range(len(eve_key)) if eve_key[i] != -1]
print(f"Alice key: {alice_key}\n")
print(f"Eve key: {eve_key}\n")
print(f"Eve correct: {correct_eve}")

Key Length: 109
Relative Error in keys 22.02%
Value of S: 1.5001585833223205
Relative Error in S: 46.96%
Number of bits of Alice key Eve knows: 44
Percentage of key known to Eve: 40.37%
Probabilty of detecting Eve: 100.00%

Alice key: [0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0]

Eve key: [0, -1, -1, -1, -1, 0, 1, -1, -1, 1, -1, -1, 1, 1, -1, 0, -1, 1, -1, -1, 1, -1, -1, -1, 0, -1, -1, 0, 0, 0, 1, -1, -1, 0, 1, -1, -1, -1, -1, 1, -1, -1, -1, -1, 1, -1, 1, 1, -1, -1, -1, 1, -1, 1, -1, -1, -1, -1, 1, 1, 0, -1, 1, -1, 0, 1, 1, 1, -1, -1, 1, -1, -1, -1, 1, 1, 1, -1, 0, -1, -1, 1, -1, -1, 1, -1, 0, 0, -1, -1, -1, 1, -1, -1, -1, -1, 1, -1, -1, -1, 1, -1, -1, -1, -1, -1, -1, 0, 0]

Eve correct: [True, True, True, T